# 04 - Deploy the Copilot Studio agent

Terraform (`infra/`) created the Azure resources and both app registrations. This notebook does the Power Platform part, which Terraform's Power Platform provider can't (it has no custom connector resource and can't publish an agent):

1. the **Workday (mock)** custom connector, with on-behalf-of (OBO) sign-in;
2. the agent, with [hr-knowledge.topic.yaml](../copilot_studio/hr-knowledge.topic.yaml), web search and general knowledge off, packaged by `pac`;
3. the agent imported into your environment;
4. the Azure AI Search connection, and the two connection references the topic uses;
5. the agent published.

Everything runs as your `az login`; no keys. Needs the Power Platform CLI to package the agent (`dotnet tool install --global Microsoft.PowerApps.CLI.Tool`, then `pac auth create --environment <your environment URL>` with the same account) and `POWER_PLATFORM_ENVIRONMENT` set through `infra/terraform.tfvars`. Rerun it after changing the topic; it updates the same connector and agent.

In [ ]:
import base64
import json
import os
import shutil
import subprocess
import time
import uuid
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

ROOT = Path.cwd() if (Path.cwd() / 'samples').is_dir() else Path.cwd().parent
load_dotenv(ROOT / '.env', override=True)
credential = AzureCliCredential(process_timeout=60)
org = os.environ['POWER_PLATFORM_ENVIRONMENT'].rstrip('/')
connector_app_id = os.environ['WORKDAY_CONNECTOR_APP_ID']
workday_host = urlparse(os.environ['WORKDAY_MOCK_URL']).hostname
agent_name, prefix = 'HR Assist', 'hrdemo'

POWER_APPS = 'https://service.powerapps.com/.default'
GRAPH = 'https://graph.microsoft.com/.default'
DATAVERSE = f'{org}/.default'

def api(method, url, scope, body=None):
    token = credential.get_token(scope).token
    # 429: the environment is busy (say, still finishing the last import) or throttling; wait and retry.
    while (response := requests.request(method, url, json=body, headers={'Authorization': f'Bearer {token}'})).status_code == 429:
        time.sleep(30)
    if not response.ok:
        raise RuntimeError(f'{method} {url} -> {response.status_code}: {response.text}')
    return response.json() if response.content else None

def run(*command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, result.stderr)
    result.check_returncode()
    return result.stdout

me = api('GET', 'https://graph.microsoft.com/v1.0/me?$select=userPrincipalName', GRAPH)['userPrincipalName']
print('Signed in as', me, '| environment', org)

## 1. Workday (mock) custom connector

Our OpenAPI file plus OAuth with Microsoft Entra ID and on-behalf-of login, using the connector app registration from Terraform. Created with the Power Apps API, the way `paconn` does it. The client secret goes from Terraform's state straight into the connector; it's never written or shown. Sign-in comes back to a redirect URL unique to this connector, which the app registration has to allow.

In [ ]:
secret = subprocess.run(['terraform', f'-chdir={ROOT / "infra"}', 'output', '-raw', 'workday_connector_client_secret'],
                        capture_output=True, text=True, check=True).stdout
environments = api('GET', 'https://api.bap.microsoft.com/providers/Microsoft.BusinessAppPlatform/environments?api-version=2021-04-01', POWER_APPS)
environment_id = next(e['name'] for e in environments['value']
                      if e['properties'].get('linkedEnvironmentMetadata', {}).get('instanceUrl', '').rstrip('/') == org)
apis = 'https://api.powerapps.com/providers/Microsoft.PowerApps/apis'
query = f"api-version=2016-11-01&$filter=environment eq '{environment_id}'"

folder = ROOT / 'copilot_studio' / 'workday-connector'
swagger = json.loads((folder / 'apiDefinition.swagger.json').read_text().replace('<WORKDAY_MOCK_HOST>', workday_host))
properties = json.loads((folder / 'apiProperties.json').read_text()
                        .replace('<CONNECTOR_APP_ID>', connector_app_id)
                        .replace('<WORKDAY_API_APP_ID>', os.environ['WORKDAY_API_APP_ID'])
                        .replace('<TENANT_ID>', os.environ['AZURE_TENANT_ID']))['properties']
properties['connectionParameters']['token']['oAuthSettings']['clientSecret'] = secret
properties.update(description=swagger['info']['description'], openApiDefinition=swagger,
                  backendService={'serviceUrl': f'https://{workday_host}/api'}, environment={'name': environment_id})

existing = next((c['name'] for c in api('GET', f'{apis}?{query}', POWER_APPS)['value']
                 if c['properties'].get('isCustomApi') and c['properties']['displayName'] == swagger['info']['title']), None)
if existing:
    api('PATCH', f'{apis}/{existing}?{query}', POWER_APPS, {'properties': properties})
else:  # the display name can only be set on create
    existing = api('POST', f'{apis}?{query}', POWER_APPS, {'properties': {**properties, 'displayName': swagger['info']['title']}})['name']
connector = api('GET', f'{apis}/{existing}?{query}', POWER_APPS)

redirect = connector['properties']['connectionParameters']['token']['oAuthSettings']['redirectUrl']
api('PATCH', f"https://graph.microsoft.com/v1.0/applications(appId='{connector_app_id}')", GRAPH, {'web': {'redirectUris': [redirect]}})
print('Connector:', connector['name'])
print('Redirect URL:', redirect)

## 2. Build the agent

`pac copilot init` scaffolds an agent on disk, we add the topic (filling in the agent's schema name and the index), turn off web search and general knowledge so answers come only from the index, and `pac copilot pack` turns it into a solution file. Nothing leaves your machine yet.

In [ ]:
workspace = ROOT / 'output' / 'copilot_agent'
shutil.rmtree(workspace, ignore_errors=True)
run('pac', 'copilot', 'init', '--name', agent_name, '--publisher-prefix', prefix, '--project-dir', str(workspace))

schema = next(line.split(':', 1)[1].strip() for line in (workspace / 'settings.mcs.yml').read_text().splitlines()
              if line.startswith('schemaName:'))
topic = ((ROOT / 'copilot_studio' / 'hr-knowledge.topic.yaml').read_text()
         .replace('<AGENT_SCHEMA>', schema).replace('<SEARCH_INDEX>', os.getenv('SEARCH_INDEX', 'hr-demo')))
header = ('mcs.metadata:\n  componentName: HR knowledge search\n'
          '  description: Looks up the user in Workday, then searches the HR index with manager / employee trimming.\n')
(workspace / 'topics' / 'HRKnowledgeSearch.mcs.yml').write_text(header + topic, encoding='utf-8')
for file, old, new in [('agent.mcs.yml', 'webBrowsing: true', 'webBrowsing: false'),
                       ('settings.mcs.yml', 'useModelKnowledge: true', 'useModelKnowledge: false')]:
    path = workspace / file
    path.write_text(path.read_text(encoding='utf-8').replace(old, new), encoding='utf-8')

run('pac', 'copilot', 'pack', '--publisher-prefix', prefix, '--project-dir', str(workspace), '--output-path', str(ROOT / 'output'))

## 3. Import it into your environment

In [ ]:
solution = (ROOT / 'output' / f'{schema}.zip').read_bytes()
api('POST', f'{org}/api/data/v9.2/ImportSolution', DATAVERSE, {
    'OverwriteUnmanagedCustomizations': True, 'PublishWorkflows': True, 'ImportJobId': str(uuid.uuid4()),
    'CustomizationFile': base64.b64encode(solution).decode()})
bot_id = api('GET', f"{org}/api/data/v9.2/bots?$select=botid&$filter=schemaname eq '{schema}'", DATAVERSE)['value'][0]['botid']
print('Imported', schema, bot_id)

## 4. Connection references

The topic calls its two connectors through connection references. A solution package carries neither the references nor their link to the topic, so we create both, as the designer does when you add a connector action:

- **Azure AI Search**: created here, signed in as the service principal Terraform made for it (`SEARCH_CONNECTION_APP_ID`), which can only read the index. Every search runs as it, so nobody has to sign in by hand. Its secret goes from Terraform's state straight into the connection.
- **Workday (mock)**: your own connection, made when you first select **Allow** in the agent. Everyone else signs in to Workday (mock) as themselves.

In [ ]:
def my_connection(connector_name):
    connections = api('GET', f'{apis}/{connector_name}/connections?{query}', POWER_APPS)['value']
    return next((c['name'] for c in connections
                 if c['properties']['createdBy'].get('userPrincipalName', '').lower() == me.lower()
                 and any(s.get('status') == 'Connected' for s in c['properties'].get('statuses', []))), None)

search_secret = subprocess.run(['terraform', f'-chdir={ROOT / "infra"}', 'output', '-raw', 'search_connection_client_secret'],
                               capture_output=True, text=True, check=True).stdout
search_connection = f"shared-azureaisearch-{uuid.uuid5(uuid.NAMESPACE_URL, os.environ['SEARCH_ENDPOINT'])}"
api('PUT', f'{apis}/shared_azureaisearch/connections/{search_connection}?{query}', POWER_APPS, {'properties': {
    'environment': {'name': environment_id},
    'displayName': f"HR search ({urlparse(os.environ['SEARCH_ENDPOINT']).hostname})",
    'connectionParametersSet': {'name': 'oauthSP', 'values': {
        'ConnectionEndpoint': {'value': os.environ['SEARCH_ENDPOINT']},
        'token': {'value': 'https://global.consent.azure-apim.net/redirect/azureaisearch'},
        'token:TenantId': {'value': os.environ['AZURE_TENANT_ID']},
        'token:clientId': {'value': os.environ['SEARCH_CONNECTION_APP_ID']},
        'token:clientSecret': {'value': search_secret}}}}})
workday_connection = my_connection(connector['name'])

component = api('GET', f"{org}/api/data/v9.2/botcomponents?$select=botcomponentid&$filter=schemaname eq '{schema}.topic.HRKnowledgeSearch'"
                '&$expand=botcomponent_connectionreference($select=connectionreferenceid)', DATAVERSE)['value'][0]
linked = [reference['connectionreferenceid'] for reference in component['botcomponent_connectionreference']]
references = {
    f'{schema}.workday-mock.get-profile': ('Workday (mock)', connector['name'], workday_connection),
    f'{schema}.shared_azureaisearch.hr-search': ('HR search (Azure AI Search)', 'shared_azureaisearch', search_connection),
}
url = f'{org}/api/data/v9.2/connectionreferences'
for name, (display_name, connector_name, connection) in references.items():
    body = {'connectionreferencedisplayname': display_name, 'connectorid': f'/providers/Microsoft.PowerApps/apis/{connector_name}'}
    if connection:
        body['connectionid'] = connection
    find = f"{url}?$select=connectionreferenceid&$filter=connectionreferencelogicalname eq '{name}'"
    found = api('GET', find, DATAVERSE)['value']
    if found:
        reference_id = found[0]['connectionreferenceid']
        api('PATCH', f'{url}({reference_id})', DATAVERSE, body)
    else:
        api('POST', url, DATAVERSE, {**body, 'connectionreferencelogicalname': name})
        reference_id = api('GET', find, DATAVERSE)['value'][0]['connectionreferenceid']
    if reference_id not in linked:
        api('POST', f"{org}/api/data/v9.2/botcomponents({component['botcomponentid']})/botcomponent_connectionreference/$ref",
            DATAVERSE, {'@odata.id': f'{url}({reference_id})'})
    print(display_name, '->', connection or 'no connection yet')

## 5. Publish

Waits for the import to finish setting up the agent, publishes, and shows the result.

In [ ]:
status_url = f'{org}/api/data/v9.2/bots({bot_id})?$select=synchronizationstatus'

def sync_status():
    return json.loads(api('GET', status_url, DATAVERSE)['synchronizationstatus'])

while sync_status()['currentSynchronizationState'].get('provisioningStatus') != 'Provisioned':
    time.sleep(10)
previous = (sync_status().get('lastFinishedPublishOperation') or {}).get('operationEnd')
api('POST', f'{org}/api/data/v9.2/bots({bot_id})/Microsoft.Dynamics.CRM.PvaPublish', DATAVERSE, {})
while (result := sync_status().get('lastFinishedPublishOperation') or {}).get('operationEnd') == previous:
    time.sleep(10)

print('Publish:', result['status'])
for detail in result.get('diagnosticDetails', []):
    for problem in detail['diagnosticList']:
        print(' ', problem['errorCode'], problem.get('errorMessage'))

In Copilot Studio, open **HR Assist** and ask "How do I prepare for a performance calibration?". The first time, select **Allow** for Workday (mock). To let others use the agent, share the Workday (mock) connector with them (Power Apps > Custom connectors > Share).